In [ ]:
# ruff: noqa: T201

import contextlib
import pathlib
import time
from collections.abc import Generator

import cache_pandas
import pandas as pd
from tqdm.auto import tqdm
from typing_extensions import Callable, Final

import svglab


tqdm.pandas()


SOURCE_DIR: Final = pathlib.Path("~/Downloads/freesvg").expanduser()
MIN_SIZE: Final = 10 * 1000  # 10 KB
MAX_SIZE: Final = 50 * 1000  # 50 KB
SAMPLES: Final = 50
SEED: Final = 42
OUTPUT: Final = pathlib.Path("output.csv")


@contextlib.contextmanager
def _timer() -> Generator[Callable[[], float]]:
    start = time.perf_counter()

    def elapsed() -> float:
        end = time.perf_counter()
        return (end - start) * 1000

    yield elapsed


def _should_include(name: str) -> bool:
    with (SOURCE_DIR / name).open("r") as f:
        try:
            svg = svglab.parse_svg(f)

            if svg.width is None or svg.height is None:
                return False

            w, h = float(svg.width), float(svg.height)
            svg.set_viewbox((0, 0, 2 * w, 2 * h))
            svg.reify()
        except Exception:  # noqa: BLE001
            return False

    return True


@cache_pandas.cache_to_csv("raw.csv")
def _load_data() -> pd.DataFrame:
    data = pd.DataFrame(
        (
            (path.name, path.stat().st_size)
            for path in SOURCE_DIR.glob("*.svg")
        ),
        columns=["filename", "size"],
    )
    return data.sort_values("filename")


@cache_pandas.cache_to_csv("good_size.csv")
def _filter_size(data: pd.DataFrame) -> pd.DataFrame:
    return data[(data["size"] > MIN_SIZE) & (data["size"] < MAX_SIZE)]


@cache_pandas.cache_to_csv("resizable.csv")
def _filter_parseable(data: pd.DataFrame) -> pd.DataFrame:
    return data[data["filename"].progress_apply(_should_include)]


@cache_pandas.cache_to_csv("sampled.csv")
def _sample_data(data: pd.DataFrame) -> pd.DataFrame:
    return data.sample(n=SAMPLES, random_state=SEED)


print("Loading SVGs...")
data = _load_data()
print(f"Loaded {len(data)} SVG files.")

print(f"Filtering SVGs by size ({MIN_SIZE} - {MAX_SIZE} bytes)...")
data = _filter_size(data)
print(f"Filtered down to {len(data)} SVG files.")

print("Filtering SVGs by parseability...")
data = _filter_parseable(data)
print(f"Filtered down to {len(data)} SVG files.")

data = _sample_data(data)

print(f"Running benchmarks on {SAMPLES} SVG files...")

for filename, _ in data.itertuples(index=False):
    with (SOURCE_DIR / filename).open("r") as f:
        with _timer() as get_elapsed:
            svg = svglab.parse_svg(f)
            data.loc[data["filename"] == filename, "parse_time"] = (
                get_elapsed()
            )

        w, h = float(svg.width or 100), float(svg.height or 100)

        with _timer() as get_elapsed:
            svg.set_viewbox((0, 0, 2 * w, 2 * h))
            svg.reify()
            data.loc[data["filename"] == filename, "reify_time"] = (
                get_elapsed()
            )

        with _timer() as get_elapsed:
            svg.get_mask(visible_only=True)
            data.loc[data["filename"] == filename, "mask_time"] = (
                get_elapsed()
            )

        with _timer() as get_elapsed:
            svg.to_xml(formatter=svglab.MINIMAL_FORMATTER)
            data.loc[data["filename"] == filename, "serialize_time"] = (
                get_elapsed()
            )

data.to_csv(OUTPUT, index=False)
